Migrated from Databricks to Neuron (auto-port).

In [ ]:
import os
import sys
from pathlib import Path

for _candidate in (
    "/home/jovyan/work/default/rashmi-ranjan/lib",
    str(Path.cwd().parent / "lib"),
    str(Path.cwd() / "lib"),
):
    if _candidate not in sys.path and Path(_candidate).is_dir():
        sys.path.insert(0, _candidate)

from neuron_runtime import (
    GoogleSheetReader,
    SlackReportsTrigger,
    get_sql_df,
    load_service_account,
    load_slack_token,
    make_unique_columns,
)

service_account = load_service_account()
slack_token = load_slack_token()

In [ ]:
import trino
import pandas as pd


query = """

WITH base_calls AS (
    SELECT
        CAST(FROM_ISO8601_TIMESTAMP(t.created_at) AS DATE) AS call_date,
        t.created_at AS call_timestamp,
        t.id AS ticket_id,
        t.uuid AS ticket_uuid,
        t.ticket_state,

        JSON_EXTRACT_SCALAR(t.meta_data, '$.userId') AS agent_email,
        JSON_EXTRACT_SCALAR(t.meta_data, '$.recordingUrl') AS recording_url,

        CASE
            WHEN cardinality(split(JSON_EXTRACT_SCALAR(t.meta_data, '$.talkTime'), ':')) = 3 THEN
                  CAST(split(JSON_EXTRACT_SCALAR(t.meta_data, '$.talkTime'), ':')[1] AS INTEGER) * 3600
                + CAST(split(JSON_EXTRACT_SCALAR(t.meta_data, '$.talkTime'), ':')[2] AS INTEGER) * 60
                + CAST(split(JSON_EXTRACT_SCALAR(t.meta_data, '$.talkTime'), ':')[3] AS INTEGER)
            WHEN cardinality(split(JSON_EXTRACT_SCALAR(t.meta_data, '$.talkTime'), ':')) = 2 THEN
                  CAST(split(JSON_EXTRACT_SCALAR(t.meta_data, '$.talkTime'), ':')[1] AS INTEGER) * 60
                + CAST(split(JSON_EXTRACT_SCALAR(t.meta_data, '$.talkTime'), ':')[2] AS INTEGER)
        END AS talk_time_seconds,

        JSON_EXTRACT_SCALAR(t.meta_data, '$.callStatus') AS call_status,
        JSON_EXTRACT_SCALAR(t.meta_data, '$.customer_data.phoneNumber') AS customer_phone,
        JSON_EXTRACT_SCALAR(t.meta_data, '$.hangupDetails') AS hangup_details,

        cb.navi_customer_id
    FROM crm_tickets.tickets t
    LEFT JOIN awsdatacatalog.gi_io_service.callback_requests cb
        ON t.uuid = cb.external_ticket_id
        AND cb.insurance_type = 'MOTOR'
    WHERE
        t.ticket_group = 'GI_MOTOR_CALLING'
        AND t.ticket_state = 'RESOLVED'
        AND JSON_EXTRACT_SCALAR(t.meta_data, '$.recordingUrl') IS NOT NULL
),

sales_data_dedup AS (
    SELECT
        customer_id AS navi_customer_id,
        policy_id   AS sale_policy_id,
        CAST(sale_date AS DATE) AS sale_date
    FROM (
        SELECT
            customer_id,
            policy_id,
            sale_date,
            ROW_NUMBER() OVER (
                PARTITION BY customer_id
                ORDER BY sale_date DESC
            ) AS rn
        FROM awsdatacatalog.corp_analytics.gi_jarvis_sale_master_motor
        WHERE customer_id IS NOT NULL
          AND is_jarvis = 1
    )
    WHERE rn = 1
),

disposition_pivot AS (
    SELECT
        da.ticket_uuid,
        MAX(CASE WHEN da.question_reference_id = 'b01f4412-ef64-49ce-a230-c6b47c4f5a0e' THEN
            CASE
                WHEN da.option_reference_ids LIKE '%5e99caad-210c-4931-b765-bd6967320152%' THEN 'Do Not Call Back'
                WHEN da.option_reference_ids LIKE '%d9f1424e-67e9-4276-ba79-1d814aa3b7b1%' THEN 'Already Purchased'
                WHEN da.option_reference_ids LIKE '%e38a34f7-17c4-48fd-9451-85897d819059%' THEN 'Will Buy Policy'
                WHEN da.option_reference_ids LIKE '%9d715ab5-0afa-42ca-8bb3-81e28b07cad7%' THEN 'Follow-Up Required'
            END
        END) AS call_outcome,
        MAX(CASE WHEN da.question_reference_id = '8f16dcf9-0ec5-4b72-a52a-c3d56e8a32f9' THEN da.answer_text END) AS dncb_reason,
        MAX(CASE WHEN da.question_reference_id = '66235230-aea7-41a1-bd35-72a0102e8c6c' THEN da.answer_text END) AS will_buy_remarks,
        MAX(CASE WHEN da.question_reference_id = '6ec63df5-62eb-471c-b030-3ca667c1fe97' THEN da.answer_text END) AS follow_up_date,
        MAX(CASE WHEN da.question_reference_id = '22718f51-2d05-49fa-b7f5-c97fdef1b360' THEN da.answer_text END) AS follow_up_time,
        MAX(CASE WHEN da.question_reference_id = '32e23845-e328-42aa-a142-7634241c04e9' THEN da.answer_text END) AS follow_up_remarks,
        MAX(CASE WHEN da.question_reference_id = 'bcd03dc5-ff14-4257-8331-0103bb9f2d97' THEN
            CASE
                WHEN da.option_reference_ids LIKE '%7f591594-fdc8-4868-ac59-9055fa4aedf2%' THEN 'High'
                WHEN da.option_reference_ids LIKE '%24f41230-8e11-4ddb-bbdc-abbe34d1742c%' THEN 'Medium'
                WHEN da.option_reference_ids LIKE '%c95aac4d-29c2-47d9-92da-101120ada5a9%' THEN 'Low'
            END
        END) AS will_buy_propensity,
        MAX(CASE WHEN da.question_reference_id = '36975d87-f6c8-4769-8918-1b822da56b0a' THEN
            CASE
                WHEN da.option_reference_ids LIKE '%fb82a0d6-78ed-45e2-b9bb-4b8940f46fbe%' THEN 'High'
                WHEN da.option_reference_ids LIKE '%80295e2c-c858-458f-a05e-80a0e734d318%' THEN 'Medium'
                WHEN da.option_reference_ids LIKE '%7c406b94-46bf-484e-a320-941d48d8d0c8%' THEN 'Low'
            END
        END) AS follow_up_propensity
    FROM awsdatacatalog.crm_tickets.disposition_answers da
    WHERE da.question_reference_id IN (
        'b01f4412-ef64-49ce-a230-c6b47c4f5a0e',
        '8f16dcf9-0ec5-4b72-a52a-c3d56e8a32f9',
        '66235230-aea7-41a1-bd35-72a0102e8c6c',
        '6ec63df5-62eb-471c-b030-3ca667c1fe97',
        '22718f51-2d05-49fa-b7f5-c97fdef1b360',
        '32e23845-e328-42aa-a142-7634241c04e9',
        'bcd03dc5-ff14-4257-8331-0103bb9f2d97',
        '36975d87-f6c8-4769-8918-1b822da56b0a'
    )
    GROUP BY da.ticket_uuid
),

final_data AS (
    SELECT
        bc.call_date,
        bc.call_timestamp,
        bc.ticket_id,
        bc.ticket_uuid,
        bc.ticket_state,
        bc.agent_email,
        bc.recording_url,
        bc.talk_time_seconds,
        bc.call_status,
        bc.customer_phone,
        bc.hangup_details,
        bc.navi_customer_id,
        sd.sale_policy_id,
        sd.sale_date,
        CASE WHEN sd.navi_customer_id IS NOT NULL THEN 'YES' ELSE 'NO' END AS sale_converted,
        CASE WHEN sd.navi_customer_id IS NOT NULL THEN 'SALE_AUDIT' ELSE 'NON_SALE_AUDIT' END AS audit_type,
        CASE
            WHEN bc.call_date = current_date - INTERVAL '1' DAY THEN 'dminus1'
            WHEN bc.call_date = current_date - INTERVAL '2' DAY THEN 'dminus2'
        END AS allocation_type,
        ROW_NUMBER() OVER (
            PARTITION BY bc.ticket_uuid
            ORDER BY bc.call_timestamp DESC
        ) AS rn,
        dp.call_outcome,
        dp.follow_up_date,
        dp.follow_up_time,
        COALESCE(dp.will_buy_remarks, dp.follow_up_remarks, dp.dncb_reason) AS remarks,
        COALESCE(dp.will_buy_propensity, dp.follow_up_propensity) AS propensity
    FROM base_calls bc
    LEFT JOIN sales_data_dedup sd
        ON bc.navi_customer_id = sd.navi_customer_id
        AND bc.call_date <= sd.sale_date
        AND (
            (
                day_of_week(current_date) = 1
                AND sd.sale_date IN (
                    current_date - INTERVAL '1' DAY,
                    current_date - INTERVAL '2' DAY
                )
            )
            OR
            (
                day_of_week(current_date) <> 1
                AND sd.sale_date = current_date - INTERVAL '1' DAY
            )
        )
    LEFT JOIN disposition_pivot dp ON bc.ticket_uuid = dp.ticket_uuid
)

SELECT *
FROM final_data
WHERE rn = 1
  AND (
        audit_type = 'SALE_AUDIT'
        OR (
            day_of_week(current_date) = 1
            AND allocation_type IN ('dminus1','dminus2')
        )
        OR (
            day_of_week(current_date) <> 1
            AND allocation_type = 'dminus1'
        )
  )
ORDER BY
    navi_customer_id,
    call_date,
    call_timestamp

"""

df  = get_sql_df(query)
if not df.empty:
    display(df)

In [ ]:
# Python cell
import os
import shutil

if not os.path.exists("data"):
    os.makedirs("data")

gsheet_url = "https://docs.google.com/spreadsheets/d/13oJSojYcn6ejjMio6c0dD39Ru-3nEJDHg21orlE4YUs/edit?gid=0#gid=0"
worksheet_name = "Master Dump"
gs_reader = GoogleSheetReader(gsheet_url, service_account)

gs_reader.clear_sheets(
    worksheet_name,
    start_row=1,
    start_column=3
)

combined_df = pd.concat([df], ignore_index=True) 

gs_reader.set_df_in_sheets( 
worksheet_name, 
combined_df, 
include_headers=True, 
row=1, 
col=3
)